In [4]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc', decode_timedelta=True) #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc', decode_timedelta=True) #***
# res='1km';t_res='5min'
# Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc', decode_timedelta=True) #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc', decode_timedelta=True) #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc', decode_timedelta=True) #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc', decode_timedelta=True) #***
# res='1km'; t_res='1min'; Np_str='100e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc', decode_timedelta=True) #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc', decode_timedelta=True) #***

In [8]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [8]:
################################################
#CALCULATING

In [4]:
dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
dir3 = dir2 + 'Project_Algorithms/Entrainment/OUTPUT/'
def initiate_array(var_names):
    # Define array dimensions (adjust based on your data)
    t_size = len(data['time'])  # Number of timesteps
    z_size = len(data['zh'])    # Number of vertical levels
    y_size = len(data['yh'])    # Number of y-axis points
    x_size = len(data['xh'])    # Number of x-axis points
    
    with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{t_res}'+'.h5', 'a') as f:
        # Check if the dataset 'theta_e' already exists
        if 'conv' not in f:
            # Create a dataset with the full size for all time steps (initially empty)
            for var_name in var_names:
                f.create_dataset(var_name, 
                                 (t_size, z_size, y_size, x_size),  # Full size for all timesteps
                                 maxshape=(None, z_size, y_size, x_size),  # Unlimited timesteps (can grow along time dimension)
                                 dtype='float64', 
                                 chunks=(1, z_size, y_size, x_size))  # Chunks for time axis to allow resizing

            
def add_timestep_at_index(timestep_data, var_name, index):
    with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{t_res}'+'.h5', 'a') as f:
        # Access the existing dataset 'theta_e'
        dataset = f[var_name]
        # Assign the new timestep data at the specified index
        dataset[index] = timestep_data

In [ ]:
var_names=['VMF_g','VMF_c']
initiate_array(var_names)

#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in range(len(data['time'])):
    if np.mod(t,1)==0: print(f'Current time {t}')

    print('loading in vars')
    w_data=data['winterp'].isel(time=t).data
    qc_data=data['qc'].isel(time=t).data
    qi_data=data['qi'].isel(time=t).data
    qcqi_data=qc_data+qi_data
    rho_data=data['rho'].isel(time=t).data

    print('calculating A_c')
    w_thresh2=0.5
    qcqi_thresh=1e-6
    cond1 = w_data >= w_thresh2
    cond2 = qcqi_data >= qcqi_thresh
    A_c = np.where(cond1 & cond2, 1, 0)

    print('calculating A_g')
    w_thresh1=0.1
    qcqi_thresh=1e-6
    cond1 = w_data >= w_thresh1
    cond2 = qcqi_data < qcqi_thresh
    A_g = np.where(cond1 & cond2, 1, 0)

    print('calculating VMF everywhere')
    VMF_c = rho_data*w_data*A_c
    VMF_g = rho_data*w_data*A_g

    print('adding VMF data timestep to H5 file')
    add_timestep_at_index(VMF_g, 'VMF_g', t)
    add_timestep_at_index(VMF_c, 'VMF_c', t)

In [11]:
################################################
#PLOTTING CONTOUR
res='1km';t_res='1min' #TESTING

In [14]:
# #READING BACK IN
# dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir3 = dir2 + 'Project_Algorithms/Entrainment/OUTPUT/'
# with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{t_res}'+'.h5', 'r') as f:
#     # VMF = f['VMF_g'][:]
#     VMF = f['VMF_c'][:]

# VMF_tz=np.mean(VMF,axis=(2,3))
# VMF_z=np.mean(VMF,axis=(0,2,3))

In [ ]:
# #USING DASK (MUCH FASTER)
# import dask.array as da
# import h5py
# from dask.diagnostics import ProgressBar

# with h5py.File(dir3 + f'3D_Eulerian_VMF_{res}_{t_res}.h5', 'r') as f:
#     dset = f['VMF_g']
#     # dset = f['VMF_c']
    
#     # Set custom chunking
#     chunks = (100, 34, 100, 256)
#     VMF = da.from_array(dset, chunks=chunks)

#     # Compute means lazily
#     VMF_tz = VMF.mean(axis=(2, 3))
#     # VMF_z = VMF.mean(axis=(0, 2, 3))
#     VMF_z = VMF_tz.mean(axis=0)

#     # Trigger computation with progress bar
#     with ProgressBar():
#         VMF_tz_result = VMF_tz.compute()
#         VMF_z_result = VMF_z.compute()

# #VMF_z_result=VMF_tz_result.mean(axis=0)

In [1]:
# import matplotlib.pyplot as plt
# from matplotlib.gridspec import GridSpec
# import numpy as np

# VMF_tz=VMF_tz_result.copy()
# VMF_z=VMF_z_result.copy()



# fig = plt.figure(figsize=(10, 8))
# gs = GridSpec(2, 2, figure=fig)

# ######
# cmap1 = plt.cm.viridis
# cmap2 = plt.cm.seismic 
# n_levels=29
# ######

# # ######
# # vmax_shared = np.max([np.max(profile_array_e), np.max(profile_array_d)])
# # norm_shared = mcolors.Normalize(vmin=0, vmax=vmax_shared)
# # ######

# # First subplot: VMF
# ########################################
# ax1 = fig.add_subplot(gs[0, 0])
# # contour1 = ax1.contourf(profile_array_e.T, cmap=cmap1)
# contour1 = ax1.contourf(VMF_tz.T, cmap=cmap1)
# cbar1=fig.colorbar(contour1, ax=ax1)
# apply_scientific_notation_colorbar([cbar1])
# Nz = len(data['zh'])
# ax1.set_yticks(np.arange(Nz))
# new_ytick_labels = np.round(data['zf'].values[:Nz], 2)
# ax1.set_yticklabels(new_ytick_labels, fontsize=8, rotation=0)
# ax1.set_ylabel('z (km)');ax1.set_xlabel('t (timesteps)')
# ax1.set_title('Eulerian VMF Profile',fontsize=8)
# plt.tight_layout()

In [2]:
# zh=data['zh'].data
# plt.plot(VMF_z,zh)
# ax=plt.gca()
# apply_scientific_notation([ax])
# plt.ylabel('z (km)');plt.xlabel('Vertical Mass Flux, M(z) '+r'($kgm^{-2}s^{-1}$)')

In [ ]:
################################################
#Taking Vertical Profiles

In [3]:
# #PROFILES AT SINGLE TIME

# t=100

# dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir3 = dir2 + 'Project_Algorithms/Entrainment/'
# with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
#     VMF = f['VMF_c'][t]

# VMF_z = np.sum(VMF,axis=(1,2))
# Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
# VMF_z/= (Nx*Ny)


# mask1 = (data['winterp'][t] >= 0.5) & ((data['qc'][t] + data['qi'][t]) >= 1e-6)
# # print(np.any(mask1))
# # masked_profile1 = np.ma.masked_where(~mask1, VMF)
# masked_profile1 = np.where(~mask1, np.nan, VMF)
# VMF_cloudy= np.nansum(masked_profile1, axis=(1, 2))
# VMF_cloudy/= (Nx*Ny)
# print('done')

# #########################


# # New Subplots for Contour Plots
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# #First plot
# axes[0].plot(VMF_z,data['zh'])
# axes[0].axvline(0,color='black')
# axes[0].set_xlabel(r"$kg m^{-2} s^{-1}$")
# axes[0].set_title('VMF(z) Horizontal Average for Eulerian Data Everywhere',fontsize=8)

# #Second plot
# axes[1].plot(VMF_cloudy,data['zh'])
# axes[1].axvline(0,color='black')
# axes[1].set_xlabel(r"$kg m^{-2} s^{-1}$")
# axes[1].set_title('VMF(z) Horizontal Average for Eulerian Data (w ≥ 0.5) & (qc+qi ≥ 1e-6)',fontsize=8)


# apply_scientific_notation([axes[0],axes[1]])
# plt.suptitle(f'Horizontal Average of 2D VMF For Entire Domain vs Within Cloudy Updrafts (AT TIME {t})')
# # plt.tight_layout()

In [4]:
# # #PROFILES AT AT ALL TIMES

# #READING BACK IN
# dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir3 = dir2 + 'Project_Algorithms/Entrainment/'
# with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
#     VMF = f['VMF_c'][:]


# VMF_z = np.sum(VMF,axis=(0,2,3))
# Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
# VMF_z/= (Ny*Nx*Nt)


# mask1 = (data['winterp'] >= 0.5) & ((data['qc'] + data['qi']) >= 1e-6)
# # print(np.any(mask1))
# # masked_profile1 = np.ma.masked_where(~mask1, VMF)
# masked_profile1 = np.where(~mask1, np.nan, VMF)
# VMF_cloudy= np.nansum(masked_profile1, axis=(0,2,3))
# VMF_cloudy/= (Ny*Nx*Nt)
# print('done')


# # ################################################


# # New Subplots for Contour Plots
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# #First plot
# axes[0].plot(VMF_z,data['zh'])
# axes[0].axvline(0,color='black')
# axes[0].set_xlabel(r"$kg m^{-2} s^{-1}$")
# axes[0].set_title('VMF(z) Horizontal Average for Eulerian Data Everywhere',fontsize=8)


# #Second plot
# axes[1].plot(VMF_cloudy,data['zh'])
# axes[1].axvline(0,color='black')
# axes[1].set_xlabel(r"$kg m^{-2} s^{-1}$")
# axes[1].set_title('VMF(z) Horizontal Average for Eulerian Data (w ≥ 0.5) & (qc+qi ≥ 1e-6)',fontsize=8)


# apply_scientific_notation([axes[0],axes[1]])
# plt.suptitle('Horizontal Average of 2D VMF For Entire Domain vs Within Cloudy Updrafts (ALL TIMES)')
# # plt.tight_layout()



In [5]:
# #PROFILES AT SINGLE TIME

# t=100

# dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir3 = dir2 + 'Project_Algorithms/Entrainment/'
# with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
#     VMF = f['VMF_g'][t]

# VMF_z = np.sum(VMF,axis=(1,2))
# Nt=len(data['time']);Ny=len(data['yh']);Nx=len(data['xh'])
# VMF_z/= (Nx*Ny)


# mask1 = (data['winterp'][t] >= 0.1) & ((data['qc'][t] + data['qi'][t]) < 1e-6)
# # print(np.any(mask1))
# # masked_profile1 = np.ma.masked_where(~mask1, VMF)
# masked_profile1 = np.where(~mask1, np.nan, VMF)
# VMF_general= np.nansum(masked_profile1, axis=(1, 2))
# VMF_general/= (Nx*Ny)
# print('done')

# #########################


# # New Subplots for Contour Plots
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# #First plot
# axes[0].plot(VMF_z,data['zh'])
# axes[0].axvline(0,color='black')
# axes[0].set_xlabel(r"$kg m^{-2} s^{-1}$")
# axes[0].set_title('VMF(z) Horizontal Average for Eulerian Data Everywhere',fontsize=8)

# #Second plot
# axes[1].plot(VMF_general,data['zh'])
# axes[1].axvline(0,color='black')
# axes[1].set_xlabel(r"$kg m^{-2} s^{-1}$")
# axes[1].set_title('VMF(z) Horizontal Average for Eulerian Data (w ≥ 0.1) & (qc+qi < 1e-6)',fontsize=8)


# apply_scientific_notation([axes[0],axes[1]])
# plt.suptitle(f'Horizontal Average of 2D VMF For Entire Domain vs Within General Updrafts (AT TIME {t})')
# # plt.tight_layout()

In [6]:
# def VMF_Profile(ax,t,data):
        
#     dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
#     dir3 = dir2 + 'Project_Algorithms/Entrainment/'
#     with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
#         VMF = f['VMF_c'][t]
    

#     Nx=512;Ny=200
#     VMF_z = np.sum(VMF,axis=(1,2))
#     VMF_z /= (Nx*Ny)

#     # #DIVIDING BY DZ (NOT NEEDED)
#     # zh=data['zh'].data*1000
#     # zf=data['zf'].data*1000
#     # for ind in np.arange(len(zh)):
#     #     dz=(zf[ind+1]-zf[ind])
#     #     VMF_z[ind] /= dz
#     #########################

    
#     #First plot
#     ax.plot(VMF_z,data['zh'])
#     ax.axvline(0,color='black')
#     ax.set_xlabel(r"$kg m^{-2} s^{-1}$")
#     ax.set_title(f'Horizontal Average of 2D VMF For Entire Domain  (AT TIME {t})',fontsize=8)
    
#     apply_scientific_notation([ax])

#     ax.set_ylim(bottom=0)
#     return axes


# fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(15, 10))
# axes = axes.flatten()
# times = np.arange(0, 130, 10)

# for idx, t in enumerate(times):
#     VMF_Profile(axes[idx], t, data)

# plt.tight_layout()
# plt.ylabel('z (km)')

In [7]:
# #TESTING DENSITY
# min=data['rho'].isel(time=100).min(dim=('zh','yh','xh')).item()
# max=data['rho'].isel(time=100).max(dim=('zh','yh','xh')).item()
# print(min,max)

# t=100
# rho_data=data['rho'].isel(time=t).data
# rho_z=np.mean(rho_data,axis=(1,2))
# plt.plot(rho_z,data['zh'])
# plt.xlabel(r'$\rho \text{ } (kg m^{-3})$')
# plt.ylabel('z (km)')

In [9]:
# #TESTING HIGH VALUE
# print('HIGH\n')

# t=100
# dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir3 = dir2 + 'Project_Algorithms/Entrainment/'
# with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
#     VMF = f['VMF'][t]


# z,y,x=np.where(VMF==VMF.max())
# VMF_val=VMF.max()
# print(f"highest VMF value is {VMF_val} at z={data['zh'][z].item():.2f} km, y={y}, x={x}\n")

# w_val=data['winterp'].isel(time=t,zh=z,yh=y,xh=x).item()
# print(f"associated w is {w_val:.2f} m/s\n")

# rho_val=data['rho'].isel(time=t,zh=z,yh=y,xh=x).item()
# print(f"associated rho is {rho_val:.2f} kg/m^2\n")

# print(f"VMF matches with (w * rho)(t,z,y,x) = {rho_val*w_val}")

In [10]:
# #TESTING LOW VALUE
# print('LOW\n')

# t=100
# dir2 = '/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir3 = dir2 + 'Project_Algorithms/Entrainment/'
# with h5py.File(dir3 + '3D_Eulerian_VMF'+f'_{res}_{Np_str}'+'.h5', 'r') as f:
#     VMF = f['VMF'][t]

# print(f"mean of VMF at time = {t} is {VMF.mean()} \n")
# cond1 = VMF<0.09
# cond2 = VMF>0.08
# z,y,x=np.where(cond1&cond2)
# z=z[0];y=y[0];x=x[0]
# VMF_val=VMF[z,y,x]
# print(f"highest VMF value is {VMF_val} at z={data['zh'][z].item():.2f} km, y={y}, x={x}\n")

# w_val=data['winterp'].isel(time=t,zh=z,yh=y,xh=x).item()
# print(f"associated w is {w_val:.2f} m/s\n")

# rho_val=data['rho'].isel(time=t,zh=z,yh=y,xh=x).item()
# print(f"associated rho is {rho_val:.2f} kg/m^2\n")

# print(f"VMF matches with (w * rho)(t,z,y,x) = {rho_val*w_val}")

In [171]:
#W HISTOGRAMS
def w_hist(w_vals, bin_num=100, w_thresh=None, ax=None):
    count_tot = len(w_vals)
    
    # Apply threshold if specified
    if w_thresh is not None:
        w_vals = w_vals[w_vals > w_thresh]
    
    # Create axis if not provided
    if ax is None:
        fig, ax = plt.subplots()
    
    # Plot histogram
    ax.hist(w_vals, bins=bin_num)
    ax.set_xlabel('w (m/s)')
    ax.set_ylabel('count')
    title = f'w > {w_thresh} m/s' if w_thresh else 'All w'
    ax.set_title(title)
    
    # Print count (optional — remove if not needed)
    print(f"[{title}] Count = {len(w_vals)}/{count_tot} = {len(w_vals)*100/count_tot:.5f}%")
    
    return ax

In [11]:
# t_slice=slice(100, 101)

# print('loading variable')
# w_vals = data['winterp'].isel(time=t_slice).data.flatten()

# print('plotting')
# # Plot with different thresholds
# t_slice = t_slice
# fig, axs = plt.subplots(2, 3, figsize=(10, 8))
# thresholds = [None, 1, 10, 20, 30, 40]

# for ax, w_thresh in zip(axs.flat, thresholds):
#     w_hist(w_vals, bin_num=100, w_thresh=w_thresh, ax=ax)

# plt.tight_layout()

In [12]:
# t_slice=slice(0,133)

# print('loading variable')
# w_vals = data['winterp'].isel(time=t_slice).data.flatten()

# print('plotting')
# # Plot with different thresholds
# t_slice = t_slice
# fig, axs = plt.subplots(2, 3, figsize=(10, 8))
# thresholds = [None, 1, 10, 20, 30, 40]

# for ax, w_thresh in zip(axs.flat, thresholds):
#     w_hist(w_vals, bin_num=100, w_thresh=w_thresh, ax=ax)

# plt.tight_layout()